In [ ]:
########################################################################
# Inclusão das Bibliotecas Necessárias
########################################################################
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
########################################################################
# Localizando o Diretório Base
########################################################################
%cd /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


In [4]:
"""
IMPLEMENTAÇÃO IDQN - VERSÃO 1.5.0
- Independent Deep Q-Network para 2 robôs
- 3000 episódios de treinamento
- Gráficos individuais separados
- Aprendizado descentralizado
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from collections import deque
import random
from typing import List, Tuple, Dict, Optional
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from datetime import datetime
import os
from pathlib import Path
import warnings
import time
import gc

warnings.filterwarnings('ignore')

# ==================== CONFIGURAÇÃO ====================
MAP_CONFIG = {
    'height': 12,
    'width': 8,
    'grid': [
        ['R1', '0', '0', '0', '0', '0', '0', 'R2'],
        ['0', 'Y', 'A', 'A', 'A', 'A', '0', '0'],
        ['0', '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['X', '0', '0', '0', 'X', '0', 'Y', '0'],
        ['0', 'Y', 'X', '0', '0', '0', '0', '0'],
        ['0', '0', '0', 'X', '0', 'Y', 'X', 'X'],
        ['0', 'X', '0', 'Y', '0', 'X', '0', '0'],
        ['0', '0', '0', 'X', '0', '0', '0', '0'],
        ['X', '0', 'Y', '0', '0', '0', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'X', '0'],
        ['X', '0', 'B', 'B', 'B', 'B', 'Y', '0'],
        ['0', '0', '0', 'Y', '0', '0', '0', '0']
    ]
}

class IDQNConfig:
    # Exploração
    EPSILON_START = 1.0
    EPSILON_END = 0.05
    EPSILON_DECAY_STEPS = 100000  # Decaimento mais longo para 3000 episódios

    # Aprendizado
    LEARNING_RATE = 0.0001
    BATCH_SIZE = 256
    GAMMA = 0.95
    TAU = 0.001
    WEIGHT_DECAY = 1e-5

    # Rede Neural
    HIDDEN_DIM = 512
    DROPOUT_RATE = 0.2

    # Memória
    BUFFER_SIZE = 100000
    PRIORITIZED_REPLAY = True
    ALPHA = 0.6
    BETA = 0.4

    # Treinamento
    MAX_GRAD_NORM = 1.0
    LEARNING_STARTS = 1000
    TRAIN_FREQ = 4
    USE_SOFT_UPDATE = True
    MAX_STEPS = 500
    EPISODES_TOTAL = 3000  # 3000 episódios

    # Sistema
    SAVE_INTERVAL = 100
    CLEAN_MEMORY_EVERY = 500

    # Falhas dos robôs
    FAILURE_PROBABILITY = 0.2
    FAILURE_PENALTY = -0.15
    ALTERNATIVE_DIRECTIONS = True


# ==================== PRIORITIZED REPLAY BUFFER ====================
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, beta=0.4):
        self.capacity = capacity
        self.alpha = alpha
        self.beta = beta
        self.buffer = []
        self.priorities = []
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        max_priority = max(self.priorities) if self.priorities else 1.0

        if len(self.buffer) < self.capacity:
            self.buffer.append((state, action, reward, next_state, done))
            self.priorities.append(max_priority)
        else:
            self.buffer[self.position] = (state, action, reward, next_state, done)
            self.priorities[self.position] = max_priority

        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        if len(self.buffer) == self.capacity:
            priorities = np.array(self.priorities)
        else:
            priorities = np.array(self.priorities[:len(self.buffer)])

        probs = priorities ** self.alpha
        probs /= probs.sum()

        indices = np.random.choice(len(self.buffer), batch_size, p=probs)

        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-self.beta)
        weights /= weights.max()

        batch = [self.buffer[idx] for idx in indices]
        states, actions, rewards, next_states, dones = zip(*batch)

        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones), indices, weights)

    def update_priorities(self, indices, td_errors):
        for idx, td_error in zip(indices, td_errors):
            self.priorities[idx] = abs(td_error) + 1e-6

    def __len__(self):
        return len(self.buffer)


# ==================== REDE NEURAL ====================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=512, dropout=0.2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        return self.network(x)


# ==================== AGENTE IDQN ====================
class IDQNAgent:
    def __init__(self, agent_id, state_dim, action_dim, config):
        self.agent_id = agent_id
        self.action_dim = action_dim
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"  Agente {agent_id} usando dispositivo: {self.device}")

        self.policy_net = DQN(state_dim, action_dim,
                              config.HIDDEN_DIM,
                              config.DROPOUT_RATE).to(self.device)
        self.target_net = DQN(state_dim, action_dim,
                              config.HIDDEN_DIM,
                              config.DROPOUT_RATE).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(),
                                   lr=config.LEARNING_RATE,
                                   weight_decay=config.WEIGHT_DECAY)

        self.memory = PrioritizedReplayBuffer(config.BUFFER_SIZE, alpha=config.ALPHA)

        self.steps_done = 0
        self.learning_steps = 0
        self.total_episodes = 0
        self.losses = []

    def get_epsilon(self):
        if self.steps_done >= self.config.EPSILON_DECAY_STEPS:
            return self.config.EPSILON_END

        epsilon = self.config.EPSILON_START - (self.config.EPSILON_START - self.config.EPSILON_END) * \
                  self.steps_done / self.config.EPSILON_DECAY_STEPS
        return max(self.config.EPSILON_END, epsilon)

    def select_action(self, state, training=True):
        self.steps_done += 1

        if training and random.random() < self.get_epsilon():
            return random.randrange(self.action_dim)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            q_values = self.policy_net(state_tensor)
            return q_values.argmax().item()

    def remember(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)

    def optimize(self):
        if len(self.memory) < self.config.BATCH_SIZE or \
           self.steps_done < self.config.LEARNING_STARTS:
            return 0

        self.learning_steps += 1

        if self.learning_steps % self.config.TRAIN_FREQ != 0:
            return 0

        states, actions, rewards, next_states, dones, indices, weights = self.memory.sample(self.config.BATCH_SIZE)
        weights = torch.FloatTensor(weights).to(self.device)

        states = torch.FloatTensor(np.array(states)).to(self.device)
        actions = torch.LongTensor(np.array(actions)).to(self.device)
        rewards = torch.FloatTensor(np.array(rewards)).to(self.device)
        next_states = torch.FloatTensor(np.array(next_states)).to(self.device)
        dones = torch.FloatTensor(np.array(dones)).to(self.device)

        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(1, keepdim=True)
            next_q_values = self.target_net(next_states).gather(1, next_actions).squeeze()
            target_q = rewards + self.config.GAMMA * next_q_values * (1 - dones)

        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()

        td_errors = target_q - current_q
        loss = (weights * td_errors.pow(2)).mean()

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), self.config.MAX_GRAD_NORM)
        self.optimizer.step()

        priorities = td_errors.abs().detach().cpu().numpy() + 1e-6
        self.memory.update_priorities(indices, priorities)

        if self.config.USE_SOFT_UPDATE:
            self.soft_update_target()

        self.losses.append(loss.item())
        return loss.item()

    def soft_update_target(self):
        for target_param, policy_param in zip(self.target_net.parameters(),
                                               self.policy_net.parameters()):
            target_param.data.copy_(self.config.TAU * policy_param.data +
                                   (1 - self.config.TAU) * target_param.data)

    def save_model(self, save_dir, episode):
        """Salva o modelo do agente"""
        os.makedirs(save_dir, exist_ok=True)
        torch.save(self.policy_net.state_dict(), save_dir / f"idqn_agent_{self.agent_id}_ep{episode}.pth")

    def load_model(self, load_dir, agent_id, episode=None):
        """Carrega o modelo do agente"""
        if episode:
            model_path = load_dir / f"idqn_agent_{agent_id}_ep{episode}.pth"
        else:
            model_path = load_dir / f"idqn_agent_{agent_id}_final.pth"

        if model_path.exists():
            self.policy_net.load_state_dict(torch.load(model_path, map_location=self.device))
            self.target_net.load_state_dict(self.policy_net.state_dict())
            return True
        return False


# ==================== AMBIENTE WAREHOUSE ====================
class WarehouseEnv(gym.Env):
    metadata = {'render.modes': ['rgb_array']}

    def __init__(self, config=None):
        super().__init__()

        self.config = config or IDQNConfig()
        self.height = MAP_CONFIG['height']
        self.width = MAP_CONFIG['width']
        self.base_grid = [row[:] for row in MAP_CONFIG['grid']]
        self.grid = [row[:] for row in self.base_grid]

        # Encontrar posições de Y
        self.y_positions = self._find_positions('Y')

        self.robot_positions = None
        self.box_positions = None
        self.targets = self._find_positions('B')

        self.num_robots = 2
        self.num_boxes = len(self._find_positions('A'))
        self.num_targets = len(self.targets)

        self.delivered_boxes = None
        self.steps = 0
        self.max_steps = self.config.MAX_STEPS
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]
        self.previous_distances = None

        self.action_space = spaces.Tuple([spaces.Discrete(6) for _ in range(self.num_robots)])

        obs_dim = (self.num_robots * 2) + (self.num_boxes * 2) + (self.num_targets * 2) + 8
        self.observation_space = spaces.Box(
            low=-1, high=self.height + self.width,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.frame_buffer = []

    def _find_positions(self, symbols):
        if isinstance(symbols, str):
            symbols = [symbols]
        positions = []
        for i in range(self.height):
            for j in range(self.width):
                cell = self.grid[i][j]
                if any(cell.startswith(sym) for sym in symbols):
                    positions.append((i, j))
        return positions

    def _remove_random_y_barriers(self):
        """Remove aleatoriamente as barreiras Y com 50% de chance"""
        self.grid = [row[:] for row in self.base_grid]

        for y_pos in self.y_positions:
            if random.random() < 0.5:
                i, j = y_pos
                self.grid[i][j] = '0'

    def reset(self):
        self._remove_random_y_barriers()

        self.robot_positions = self._find_positions('R')
        self.box_positions = self._find_positions('A')
        self.delivered_boxes = [False] * self.num_boxes
        self.targets = self._find_positions('B')

        self.steps = 0
        self.total_deliveries = 0
        self.collisions = 0
        self.failures = [0, 0]
        self.distance_traveled = [0, 0]

        self.previous_distances = [self._min_distance_to_boxes(r) for r in range(self.num_robots)]

        return self._get_observation(), self._get_info()

    def _min_distance_to_boxes(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        remaining_boxes = [box_pos for box_pos in self.box_positions
                          if box_pos is not None and
                          not self.delivered_boxes[self.box_positions.index(box_pos)]]

        if not remaining_boxes:
            return 0
        return min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                   for box_pos in remaining_boxes])

    def _is_valid_position(self, pos, robot_id=None):
        i, j = pos
        if i < 0 or i >= self.height or j < 0 or j >= self.width:
            return False

        cell = self.grid[i][j]
        if cell in ['X', 'Y']:
            return False

        if robot_id is not None:
            for rid, rpos in enumerate(self.robot_positions):
                if rid != robot_id and rpos == (i, j):
                    return False
        return True

    def _get_direction_name(self, action):
        direcoes = {0: "CIMA", 1: "BAIXO", 2: "ESQUERDA", 3: "DIREITA", 4: "PEGAR", 5: "SOLTAR"}
        return direcoes.get(action, "DESCONHECIDO")

    def _get_alternative_direction(self, original_action, robot_id):
        i, j = self.robot_positions[robot_id]
        alternative_actions = [a for a in range(4) if a != original_action]
        random.shuffle(alternative_actions)

        for alt_action in alternative_actions:
            alt_i, alt_j = i, j
            if alt_action == 0: alt_i -= 1
            elif alt_action == 1: alt_i += 1
            elif alt_action == 2: alt_j -= 1
            elif alt_action == 3: alt_j += 1

            if self._is_valid_position((alt_i, alt_j), robot_id):
                return alt_action, (alt_i, alt_j)

        return None, None

    def _move_robot_with_failure(self, robot_id, action):
        i, j = self.robot_positions[robot_id]
        new_i, new_j = i, j

        if action == 0: new_i -= 1
        elif action == 1: new_i += 1
        elif action == 2: new_j -= 1
        elif action == 3: new_j += 1

        desired_valid = self._is_valid_position((new_i, new_j), robot_id)

        if random.random() < self.config.FAILURE_PROBABILITY:
            self.failures[robot_id] += 1

            if self.config.ALTERNATIVE_DIRECTIONS:
                alt_action, alt_pos = self._get_alternative_direction(action, robot_id)
                if alt_action is not None:
                    old_pos = self.robot_positions[robot_id]
                    self.grid[old_pos[0]][old_pos[1]] = '0'
                    self.grid[alt_pos[0]][alt_pos[1]] = f'R{robot_id + 1}'
                    self.robot_positions[robot_id] = alt_pos
                    self.distance_traveled[robot_id] += 1
                    return self.config.FAILURE_PENALTY - 0.01

            return self.config.FAILURE_PENALTY

        if desired_valid:
            self.distance_traveled[robot_id] += 1
            old_pos = self.robot_positions[robot_id]
            self.grid[old_pos[0]][old_pos[1]] = '0'
            self.grid[new_i][new_j] = f'R{robot_id + 1}'
            self.robot_positions[robot_id] = (new_i, new_j)
            return -0.005
        else:
            self.collisions += 1
            return -0.02

    def _pickup_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]
        for box_id, box_pos in enumerate(self.box_positions):
            if not self.delivered_boxes[box_id] and box_pos == robot_pos:
                self.box_positions[box_id] = None
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'
                return 2.0
        return -0.02

    def _drop_box(self, robot_id):
        robot_pos = self.robot_positions[robot_id]

        box_with_robot = None
        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None and not self.delivered_boxes[box_id]:
                box_with_robot = box_id
                break

        if box_with_robot is None:
            return -0.02

        for target_pos in self.targets:
            if robot_pos == target_pos:
                self.delivered_boxes[box_with_robot] = True
                self.total_deliveries += 1
                self.grid[robot_pos[0]][robot_pos[1]] = f'R{robot_id + 1}'
                return 25.0

        return -0.05

    def _calculate_shaped_reward(self, robot_id, base_reward):
        reward = base_reward

        current_distance = self._min_distance_to_boxes(robot_id)
        previous_distance = self.previous_distances[robot_id]

        if current_distance < previous_distance:
            reward += 0.1 * (previous_distance - current_distance)
        elif current_distance > previous_distance:
            reward -= 0.02 * (current_distance - previous_distance)

        self.previous_distances[robot_id] = current_distance

        if all(self.delivered_boxes):
            reward += 50.0

        return reward

    def _get_observation(self):
        obs = []

        for robot_pos in self.robot_positions:
            obs.append(robot_pos[0] / self.height)
            obs.append(robot_pos[1] / self.width)

        for box_id, box_pos in enumerate(self.box_positions):
            if box_pos is None or self.delivered_boxes[box_id]:
                obs.append(-1)
                obs.append(-1)
            else:
                obs.append(box_pos[0] / self.height)
                obs.append(box_pos[1] / self.width)

        for target_pos in self.targets:
            obs.append(target_pos[0] / self.height)
            obs.append(target_pos[1] / self.width)

        for robot_pos in self.robot_positions:
            min_box_dist = min([abs(robot_pos[0] - box_pos[0]) + abs(robot_pos[1] - box_pos[1])
                               for box_pos in self.box_positions
                               if box_pos is not None and
                               not self.delivered_boxes[self.box_positions.index(box_pos)]],
                              default=100)
            obs.append(min_box_dist / (self.height + self.width))

            min_target_dist = min([abs(robot_pos[0] - target_pos[0]) + abs(robot_pos[1] - target_pos[1])
                                  for target_pos in self.targets],
                                 default=100)
            obs.append(min_target_dist / (self.height + self.width))

        return np.array(obs, dtype=np.float32)

    def step(self, actions):
        self.steps += 1

        if len(actions) != self.num_robots:
            actions = [actions] * self.num_robots

        total_reward = 0
        rewards = [0, 0]

        # Movimentos
        movement_rewards = []
        for robot_id, action in enumerate(actions):
            if action < 4:
                reward = self._move_robot_with_failure(robot_id, action)
                movement_rewards.append(reward)
            else:
                movement_rewards.append(0)

        # Interações
        interaction_rewards = []
        for robot_id, action in enumerate(actions):
            if action == 4:
                reward = self._pickup_box(robot_id)
                interaction_rewards.append(reward)
            elif action == 5:
                reward = self._drop_box(robot_id)
                interaction_rewards.append(reward)
            else:
                interaction_rewards.append(0)

        for robot_id in range(self.num_robots):
            base_reward = movement_rewards[robot_id] + interaction_rewards[robot_id]
            shaped_reward = self._calculate_shaped_reward(robot_id, base_reward)
            rewards[robot_id] = shaped_reward
            total_reward += shaped_reward

        terminated = all(self.delivered_boxes)
        truncated = self.steps >= self.max_steps

        observation = self._get_observation()
        info = self._get_info()

        return observation, rewards, terminated, truncated, info

    def _get_info(self):
        return {
            'steps': self.steps,
            'total_deliveries': self.total_deliveries,
            'collisions': self.collisions,
            'failures_r1': self.failures[0],
            'failures_r2': self.failures[1],
            'distance_traveled': self.distance_traveled.copy(),
            'remaining_boxes': sum(1 for d in self.delivered_boxes if not d),
            'success_rate': self.total_deliveries / self.num_boxes if self.steps > 0 else 0,
        }

    def close(self):
        pass


# ==================== FUNÇÕES DE PLOTAGEM ====================
def plot_individual_graphs(metrics, save_dir):
    """Plota gráficos individuais separados"""

    print(f"\n📊 Gerando gráficos individuais em: {save_dir}")

    episodes = range(1, len(metrics['episode_rewards']) + 1)

    # 1. GRÁFICO DE RECOMPENSA
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['episode_rewards'], 'b-', alpha=0.5, linewidth=1)
    if len(metrics['episode_rewards']) >= 100:
        moving_avg = np.convolve(metrics['episode_rewards'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_rewards']) + 1), moving_avg, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Recompensa', fontsize=12)
    plt.title('Recompensa por Episódio - IDQN (3000 episódios)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_recompensa_idqn.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_recompensa_idqn.png salvo")

    # 2. GRÁFICO DE ENTREGAS
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['episode_deliveries'], 'g-', alpha=0.5, linewidth=1)
    if len(metrics['episode_deliveries']) >= 100:
        moving_avg_del = np.convolve(metrics['episode_deliveries'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_deliveries']) + 1), moving_avg_del, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.axhline(y=8, color='orange', linestyle='--', linewidth=2, label='Meta (8 caixas)')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Entregas', fontsize=12)
    plt.title('Entregas por Episódio - IDQN (3000 episódios)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_entregas_idqn.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_entregas_idqn.png salvo")

    # 3. GRÁFICO DE STEPS
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['episode_steps'], 'orange', alpha=0.5, linewidth=1)
    if len(metrics['episode_steps']) >= 100:
        moving_avg_steps = np.convolve(metrics['episode_steps'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['episode_steps']) + 1), moving_avg_steps, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Steps', fontsize=12)
    plt.title('Steps por Episódio - IDQN (3000 episódios)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_steps_idqn.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_steps_idqn.png salvo")

    # 4. GRÁFICO DE TAXA DE SUCESSO
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['success_rates'], 'purple', alpha=0.5, linewidth=1)
    if len(metrics['success_rates']) >= 100:
        moving_avg_success = np.convolve(metrics['success_rates'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['success_rates']) + 1), moving_avg_success, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.axhline(y=0.95, color='green', linestyle='--', linewidth=2, label='Meta 95%')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Taxa de Sucesso', fontsize=12)
    plt.title('Taxa de Sucesso por Episódio - IDQN (3000 episódios)', fontsize=14, fontweight='bold')
    plt.ylim([0, 1.05])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_taxa_sucesso_idqn.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_taxa_sucesso_idqn.png salvo")

    # 5. GRÁFICO DE COLISÕES
    plt.figure(figsize=(14, 7))
    plt.plot(episodes, metrics['collisions'], 'red', alpha=0.5, linewidth=1)
    if len(metrics['collisions']) >= 100:
        moving_avg_coll = np.convolve(metrics['collisions'], np.ones(100)/100, mode='valid')
        plt.plot(range(100, len(metrics['collisions']) + 1), moving_avg_coll, 'r-', linewidth=2, label='Média Móvel (100)')
    plt.xlabel('Episódio', fontsize=12)
    plt.ylabel('Colisões', fontsize=12)
    plt.title('Colisões por Episódio - IDQN (3000 episódios)', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / 'grafico_colisoes_idqn.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_colisoes_idqn.png salvo")

    # 6. GRÁFICO DE FALHAS
    if 'failures_r1' in metrics and metrics['failures_r1']:
        plt.figure(figsize=(14, 7))
        failures_total = np.array(metrics['failures_r1']) + np.array(metrics['failures_r2'])
        plt.plot(episodes, failures_total, 'brown', alpha=0.5, linewidth=1, label='Total')
        plt.plot(episodes, metrics['failures_r1'], 'red', alpha=0.5, linewidth=1, label='R1')
        plt.plot(episodes, metrics['failures_r2'], 'blue', alpha=0.5, linewidth=1, label='R2')
        if len(failures_total) >= 100:
            moving_avg_fail = np.convolve(failures_total, np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(failures_total) + 1), moving_avg_fail, 'r-', linewidth=2, label='Média Móvel Total')
        plt.xlabel('Episódio', fontsize=12)
        plt.ylabel('Falhas', fontsize=12)
        plt.title('Falhas por Episódio - IDQN (3000 episódios)', fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_dir / 'grafico_falhas_idqn.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_falhas_idqn.png salvo")

    # 7. GRÁFICO DE DISTÂNCIA PERCORRIDA
    if 'distance_traveled' in metrics and metrics['distance_traveled']:
        plt.figure(figsize=(14, 7))
        plt.plot(episodes, metrics['distance_traveled'], 'teal', alpha=0.5, linewidth=1)
        if len(metrics['distance_traveled']) >= 100:
            moving_avg_dist = np.convolve(metrics['distance_traveled'], np.ones(100)/100, mode='valid')
            plt.plot(range(100, len(metrics['distance_traveled']) + 1), moving_avg_dist, 'r-', linewidth=2, label='Média Móvel (100)')
        plt.xlabel('Episódio', fontsize=12)
        plt.ylabel('Distância Percorrida', fontsize=12)
        plt.title('Distância Percorrida por Episódio - IDQN (3000 episódios)', fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_dir / 'grafico_distancia_idqn.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_distancia_idqn.png salvo")

    print(f"📊 Todos os gráficos foram salvos em: {save_dir}")


def save_metrics_csv(metrics, save_dir):
    """Salva as métricas em arquivo CSV"""
    df = pd.DataFrame({
        'episodio': range(1, len(metrics['episode_rewards']) + 1),
        'recompensa': metrics['episode_rewards'],
        'entregas': metrics['episode_deliveries'],
        'steps': metrics['episode_steps'],
        'taxa_sucesso': metrics['success_rates'],
        'colisoes': metrics['collisions'],
        'falha_r1': metrics['failures_r1'],
        'falha_r2': metrics['failures_r2'],
        'distancia_percorrida': metrics.get('distance_traveled', [0] * len(metrics['episode_rewards']))
    })
    df.to_csv(save_dir / 'metricas_treinamento_idqn.csv', index=False)
    print(f"📊 Métricas salvas em: {save_dir / 'metricas_treinamento_idqn.csv'}")


# ==================== FUNÇÃO PRINCIPAL DE TREINAMENTO IDQN ====================
def run_idqn_training(num_episodes=3000):
    """Executa treinamento IDQN por 3000 episódios"""

    config = IDQNConfig()
    config.EPISODES_TOTAL = num_episodes

    print("=" * 80)
    print("🤖 TREINAMENTO IDQN - 3000 EPISÓDIOS")
    print("=" * 80)
    print(f"\n📋 CONFIGURAÇÃO:")
    print(f"   • Modelo: IDQN (Independent Deep Q-Network)")
    print(f"   • Dispositivo: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
    print(f"   • Total de episódios: {num_episodes}")
    print(f"   • Máximo de steps por episódio: {config.MAX_STEPS}")
    print(f"   • Batch Size: {config.BATCH_SIZE}")
    print(f"   • Learning Rate: {config.LEARNING_RATE}")
    print(f"   • Hidden Dim: {config.HIDDEN_DIM}")
    print(f"   • Falhas nos robôs: {config.FAILURE_PROBABILITY*100:.0f}% chance")
    print(f"   • Barreiras Y removidas aleatoriamente (50% chance)")
    print("=" * 80)

    # Inicializar ambiente para obter dimensões
    env = WarehouseEnv(config=config)
    sample_obs, _ = env.reset()
    state_dim = len(sample_obs)
    action_dim = 6
    env.close()

    print(f"\n📊 DIMENSÕES:")
    print(f"   • Estado individual: {state_dim}")
    print(f"   • Ações: {action_dim}")
    print(f"   • Robôs: 2")
    print(f"   • Caixas: 8")

    # Criar agentes IDQN
    agents = [IDQNAgent(i, state_dim, action_dim, config) for i in range(2)]

    metrics = {
        'episode_rewards': [],
        'episode_deliveries': [],
        'episode_steps': [],
        'success_rates': [],
        'collisions': [],
        'failures_r1': [],
        'failures_r2': [],
        'distance_traveled': []
    }

    # Criar diretório de resultados
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path(f"idqn_results_{timestamp}")
    results_dir.mkdir(exist_ok=True)
    print(f"\n📁 Diretório de resultados: {results_dir.absolute()}")

    total_start_time = time.time()

    for episode in range(num_episodes):
        obs, _ = env.reset()
        episode_reward = 0
        episode_collisions = 0

        for step in range(config.MAX_STEPS):
            # Selecionar ações
            actions = [agent.select_action(obs, training=True) for agent in agents]

            # Executar ações
            next_obs, rewards, terminated, truncated, info = env.step(actions)

            # Armazenar experiências e otimizar
            for i, agent in enumerate(agents):
                agent.remember(obs, actions[i], rewards[i], next_obs, terminated or truncated)
                episode_reward += rewards[i]

            episode_collisions = info['collisions']

            # Otimizar agentes
            for agent in agents:
                agent.optimize()

            obs = next_obs

            if terminated or truncated:
                break

        # Registrar métricas
        metrics['episode_rewards'].append(episode_reward)
        metrics['episode_deliveries'].append(info['total_deliveries'])
        metrics['episode_steps'].append(step + 1)
        metrics['success_rates'].append(info['success_rate'])
        metrics['collisions'].append(episode_collisions)
        metrics['failures_r1'].append(info['failures_r1'])
        metrics['failures_r2'].append(info['failures_r2'])
        metrics['distance_traveled'].append(sum(info['distance_traveled']))

        # Atualizar total de episódios dos agentes
        for agent in agents:
            agent.total_episodes = episode + 1

        # Logging a cada 100 episódios
        if (episode + 1) % 100 == 0:
            recent_rewards = metrics['episode_rewards'][-100:]
            recent_deliveries = metrics['episode_deliveries'][-100:]
            epsilon = agents[0].get_epsilon()
            elapsed = time.time() - total_start_time

            print(f"Ep {episode+1:4d}/{num_episodes} | "
                  f"Reward: {np.mean(recent_rewards):7.2f} | "
                  f"Entregas: {np.mean(recent_deliveries):.2f}/8 | "
                  f"ε: {epsilon:.3f} | "
                  f"Tempo: {elapsed/60:.1f}min")

        # Salvar modelos periodicamente
        if (episode + 1) % 500 == 0:
            models_dir = results_dir / "models"
            for agent in agents:
                agent.save_model(models_dir, episode + 1)

        # Limpar memória periodicamente
        if (episode + 1) % config.CLEAN_MEMORY_EVERY == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    total_time = (time.time() - total_start_time) / 60
    env.close()

    print(f"\n💾 SALVANDO RESULTADOS...")

    # Salvar métricas
    save_metrics_csv(metrics, results_dir)

    # Plotar gráficos individuais
    plot_individual_graphs(metrics, results_dir)

    # Salvar modelos finais
    models_dir = results_dir / "models"
    models_dir.mkdir(exist_ok=True)
    for agent in agents:
        torch.save(agent.policy_net.state_dict(), models_dir / f"idqn_agent_{agent.agent_id}_final.pth")

    # Calcular estatísticas finais
    final_deliveries = metrics['episode_deliveries'][-100:]
    final_rewards = metrics['episode_rewards'][-100:]
    final_collisions = metrics['collisions'][-100:]

    # Relatório final
    report = f"""
    ========================================
    RELATÓRIO FINAL - IDQN (3000 EPISÓDIOS)
    ========================================

    DATA: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    DIRETÓRIO: {results_dir.absolute()}

    CONFIGURAÇÃO:
    - Modelo: IDQN (Independent Deep Q-Network)
    - Total de episódios: {num_episodes}
    - Tempo total: {total_time:.1f} minutos
    - Batch Size: {config.BATCH_SIZE}
    - Learning Rate: {config.LEARNING_RATE}
    - Hidden Dim: {config.HIDDEN_DIM}
    - Falhas nos robôs: {config.FAILURE_PROBABILITY*100:.0f}% chance
    - Barreiras Y: removidas aleatoriamente (50% chance)

    MÉTRICAS FINAIS (últimos 100 episódios):
    - Recompensa média: {np.mean(final_rewards):.2f}
    - Entregas médias: {np.mean(final_deliveries):.2f}/8
    - Taxa de sucesso final: {metrics['success_rates'][-1]:.1%}
    - Colisões médias: {np.mean(final_collisions):.1f}
    - Total falhas R1: {sum(metrics['failures_r1'])}
    - Total falhas R2: {sum(metrics['failures_r2'])}

    MELHORES RESULTADOS:
    - Melhor recompensa: {max(metrics['episode_rewards']):.2f}
    - Melhor entrega: {max(metrics['episode_deliveries'])}/8
    - Menor número de steps: {min(metrics['episode_steps'])}

    GRÁFICOS GERADOS:
    - grafico_recompensa_idqn.png
    - grafico_entregas_idqn.png
    - grafico_steps_idqn.png
    - grafico_taxa_sucesso_idqn.png
    - grafico_colisoes_idqn.png
    - grafico_falhas_idqn.png
    - grafico_distancia_idqn.png

    ========================================
    """

    with open(results_dir / "relatorio_final_idqn.txt", 'w', encoding='utf-8') as f:
        f.write(report)

    print(report)
    print(f"\n✅ TREINAMENTO IDQN CONCLUÍDO!")
    print(f"📁 Resultados salvos em: {results_dir.absolute()}")
    print(f"   - metricas_treinamento_idqn.csv")
    print(f"   - grafico_recompensa_idqn.png")
    print(f"   - grafico_entregas_idqn.png")
    print(f"   - grafico_steps_idqn.png")
    print(f"   - grafico_taxa_sucesso_idqn.png")
    print(f"   - grafico_colisoes_idqn.png")
    print(f"   - grafico_falhas_idqn.png")
    print(f"   - grafico_distancia_idqn.png")
    print(f"   - relatorio_final_idqn.txt")
    print(f"   - models/ (modelos treinados)")

    return agents, metrics, results_dir


# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    NUM_EPISODES = 3000

    print("\n" + "=" * 80)
    print("🎮 INICIANDO TREINAMENTO IDQN")
    print("=" * 80)
    print("\n⚠️ CARACTERÍSTICAS DO IDQN:")
    print("   • Cada robô tem sua própria rede neural")
    print("   • Aprendizado descentralizado")
    print("   • Memória de replay prioritizada")
    print("   • Target networks para estabilidade")
    print("   • 3000 episódios completos")
    print("   • Ambiente com barreiras Y removíveis")
    print("   • Falhas nos movimentos (20%)\n")

    try:
        agents, metrics, results_dir = run_idqn_training(num_episodes=NUM_EPISODES)

        print("\n" + "=" * 60)
        print("✨ TREINAMENTO IDQN CONCLUÍDO COM SUCESSO! ✨")
        print("=" * 60)

        # Mostrar estatísticas resumidas
        print("\n📊 ESTATÍSTICAS FINAIS DO IDQN:")
        print(f"   • Recompensa média (últimos 100): {np.mean(metrics['episode_rewards'][-100:]):.2f}")
        print(f"   • Entregas médias (últimos 100): {np.mean(metrics['episode_deliveries'][-100:]):.2f}/8")
        print(f"   • Taxa de sucesso final: {metrics['success_rates'][-1]:.1%}")
        print(f"   • Colisões médias (últimos 100): {np.mean(metrics['collisions'][-100:]):.1f}")

    except Exception as e:
        print(f"\n❌ ERRO DURANTE O TREINAMENTO: {e}")
        import traceback
        traceback.print_exc()


🎮 INICIANDO TREINAMENTO IDQN

⚠️ CARACTERÍSTICAS DO IDQN:
   • Cada robô tem sua própria rede neural
   • Aprendizado descentralizado
   • Memória de replay prioritizada
   • Target networks para estabilidade
   • 3000 episódios completos
   • Ambiente com barreiras Y removíveis
   • Falhas nos movimentos (20%)

🤖 TREINAMENTO IDQN - 3000 EPISÓDIOS

📋 CONFIGURAÇÃO:
   • Modelo: IDQN (Independent Deep Q-Network)
   • Dispositivo: CUDA
   • Total de episódios: 3000
   • Máximo de steps por episódio: 500
   • Batch Size: 256
   • Learning Rate: 0.0001
   • Hidden Dim: 512
   • Falhas nos robôs: 20% chance
   • Barreiras Y removidas aleatoriamente (50% chance)

📊 DIMENSÕES:
   • Estado individual: 40
   • Ações: 6
   • Robôs: 2
   • Caixas: 8
  Agente 0 usando dispositivo: cuda
  Agente 1 usando dispositivo: cuda

📁 Diretório de resultados: /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código/idqn_results_20260612_005336
Ep  100/3000 | Reward:  173.79 | Entregas: 5.67/8 | ε: 0